In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import random
import os
import gc
import matplotlib.pyplot as plt

def set_seed(seed=42):
    print("Minden véletlenszám-generátor fixálása a reprodukálhatóságért.")
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

from Option_func import (
    HestonAsianPINN, PINNScaler, OptionSampler,  
    calculate_heston_asian_pde, calculate_s_zero_loss, 
    calculate_payoff_loss, calculate_neumann_bc_loss
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

params = {
    'r': 0.05,
    'kappa': 2.0,
    'theta': 0.1,
    'sigma': 0.3,
    'rho': -0.7,
    'K': 1.55
}

T_maturity = 1.0
num_epochs = 3000

batch_size_interior = 25000  
batch_size_ic = 5000         
batch_size_bc = 2500         

scaler = PINNScaler(t_max=T_maturity, S_min=0.0, S_max=5.0, 
                    v_min=0.0001, v_max=1.0, A_min=0.0, A_max=5.0) 

sampler = OptionSampler(device=device, tau_max=T_maturity, S_min=0.0, S_max=5.0,
                        v_min=0.0001, v_max=1.0, A_min=0.0, A_max=5.0) # 

model = HestonAsianPINN(scaler=scaler).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=500, factor=0.5)

# A lambdákat előre hoztuk, hogy a try blokk már ismerje a lambdas['mc']-t
lambdas = {
    'pde': 10.0,
    'mc': 1.0,      
    'ic': 7.0,
    's0': 1.0,
    'neumann': 1.0
}

try:
    anchors = torch.load('heston_anchors.pth', weights_only=False)
    S_mc = torch.tensor(anchors['S0'], dtype=torch.float32).reshape(-1, 1).to(device).requires_grad_(True)
    v_mc = torch.tensor(anchors['v0'], dtype=torch.float32).reshape(-1, 1).to(device).requires_grad_(True)
    U_mc = torch.tensor(anchors['prices'], dtype=torch.float32).reshape(-1, 1).to(device)
    
    A_mc = torch.zeros_like(S_mc).to(device)
    tau_mc = torch.full_like(S_mc, T_maturity).to(device)
    print(f"Monte Carlo adatok betöltve: {len(U_mc)} db")
    
   
    mc_weights = torch.ones_like(S_mc).to(device) * lambdas['mc']
    mask_strike_mc = (S_mc > 1.0) & (S_mc < 2.1)
    mc_weights[mask_strike_mc] = 30.0
    
except Exception as e:
    print(f"Figyelem: Nem sikerült betölteni a heston_anchors.pth fájlt! Hiba: {e}")
    S_mc = torch.zeros((1,1)).to(device).requires_grad_(True)
    v_mc = torch.zeros((1,1)).to(device).requires_grad_(True)
    U_mc, A_mc, tau_mc = [torch.zeros((1,1)).to(device) for _ in range(3)]
    
    
    mc_weights = torch.ones((1,1)).to(device) * lambdas['mc']

loss_history = []
epoch_history = []
total_points_seen = 0

print(f"\nIndul a dinamikus tanítás! Minden epochban {batch_size_interior} belső pont generálódik.")

#Tanítás
for epoch in range(num_epochs):
    model.train() 
    optimizer.zero_grad()
    
    if epoch < 1000:
        current_lambda_pde = 0.0
    else:
        current_lambda_pde = lambdas['pde'] 
    
    #  új pontok sorsolása a sampler .py-ben(OptionSampler osztály) lévő függvényekből
    t_in, S_in, v_in, A_in = sampler.sample_interior(batch_size_interior)
    t_ic, S_ic, v_ic, A_ic = sampler.sample_initial_condition(batch_size_ic)
    t_b0, S_b0, v_b0, A_b0 = sampler.sample_boundary_S(batch_size_bc, 0.0)
    t_bmax, S_bmax, v_bmax, A_bmax = sampler.sample_boundary_S(batch_size_bc, sampler.S_max)
    
    t_in.requires_grad_(True); S_in.requires_grad_(True); v_in.requires_grad_(True); A_in.requires_grad_(True)
    t_bmax.requires_grad_(True); S_bmax.requires_grad_(True); v_bmax.requires_grad_(True); A_bmax.requires_grad_(True)
    
    # Loss  Calculation
     #Ha a lambda 0,  nem számol feleslegesen deriváltakat
    if current_lambda_pde > 0.0:
        res_pde = calculate_heston_asian_pde(model, t_in, S_in, v_in, A_in, params)
        
       
        pde_weights = torch.ones_like(S_in)
        mask_pde = (S_in > 1.0) & (S_in < 2.1) & (A_in < 1.6)
        pde_weights[mask_pde] = 20.0
        
        loss_pde = torch.mean(pde_weights * res_pde**2)
    else:
        loss_pde = torch.tensor(0.0, device=device)
    
    U_pred_mc = model(tau_mc, S_mc, v_mc, A_mc)
    
   
    loss_mc = torch.mean(mc_weights * (U_pred_mc - U_mc)**2)
    
    loss_ic = calculate_payoff_loss(model, t_ic, S_ic, v_ic, A_ic, T_maturity, params)
    loss_s0 = calculate_s_zero_loss(model, t_b0, v_b0, A_b0, params, T_maturity)
    loss_neumann = calculate_neumann_bc_loss(model, t_bmax, S_bmax, v_bmax, A_bmax)

    # A current_lambda_pde-t kell használni!
   
    total_loss = (current_lambda_pde * loss_pde + 
                  loss_mc + 
                  lambdas['ic'] * loss_ic + 
                  lambdas['s0'] * loss_s0 + 
                  lambdas['neumann'] * loss_neumann)
    #Backprop 
    total_loss.backward()
    #Theta frissításe, azaz a mátrixban lévő értékek
    optimizer.step()
    #scheduler, h módosítsa-e a lr-t. .item, hogy magát az értéket vegye csak
    scheduler.step(total_loss.item())
    
    total_points_seen += batch_size_interior

    if epoch % 100 == 0:
        loss_history.append(total_loss.item())
        epoch_history.append(epoch)
        phase_name = "Forma tanulása (Nincs PDE)" if epoch < 1000 else "Fizika finomhangolása (PDE ON)"
        print(f"Epoch {epoch:4d} [{phase_name}] | Loss: {total_loss.item():.6e} | PDE: {loss_pde.item():.6e} | MC: {loss_mc.item():.6e} | IC: {loss_ic.item():.6e} | S0: {loss_s0.item():.6e} | Neumann: {loss_neumann.item():.6e}")

print(f"\n✓ Adam kész! Összesen {total_points_seen:,} egyedi pontot vizsgáltunk meg.")
print("Indul az L-BFGS finomhangolás egy fix, 50 000 pontos masszív hálón...")

t_in, S_in, v_in, A_in = sampler.sample_interior(50000)
t_ic, S_ic, v_ic, A_ic = sampler.sample_initial_condition(10000)
t_b0, S_b0, v_b0, A_b0 = sampler.sample_boundary_S(5000, 0.0)
t_bmax, S_bmax, v_bmax, A_bmax = sampler.sample_boundary_S(5000, sampler.S_max)

t_in.requires_grad_(True); S_in.requires_grad_(True); v_in.requires_grad_(True); A_in.requires_grad_(True)
t_ic.requires_grad_(True); S_ic.requires_grad_(True); v_ic.requires_grad_(True); A_ic.requires_grad_(True)
t_b0.requires_grad_(True); v_b0.requires_grad_(True); A_b0.requires_grad_(True)
t_bmax.requires_grad_(True); S_bmax.requires_grad_(True); v_bmax.requires_grad_(True); A_bmax.requires_grad_(True)

l_optimizer = torch.optim.LBFGS(
    model.parameters(), lr=1.0, max_iter=10000, tolerance_grad=1e-7, tolerance_change=1e-12, line_search_fn="strong_wolfe"
)

lbfgs_iter = 0

def closure():
    global lbfgs_iter
    l_optimizer.zero_grad()
    
    res = calculate_heston_asian_pde(model, t_in, S_in, v_in, A_in, params)
    
    
    pde_weights = torch.ones_like(S_in)
    mask_pde = (S_in > 1.0) & (S_in < 2.1) & (A_in < 1.6) 
    pde_weights[mask_pde] = 80.0
    
    l_pde = torch.mean(pde_weights * res**2)
    
    U_pred_mc_lbfgs = model(tau_mc, S_mc, v_mc, A_mc)
    
   
    l_mc = torch.mean(mc_weights * (U_pred_mc_lbfgs - U_mc)**2)
    
    l_ic = calculate_payoff_loss(model, t_ic, S_ic, v_ic, A_ic, T_maturity, params)
    l_s0 = calculate_s_zero_loss(model, t_b0, v_b0, A_b0, params, T_maturity)
    l_neu = calculate_neumann_bc_loss(model, t_bmax, S_bmax, v_bmax, A_bmax)
    
    
    t_loss = (lambdas['pde'] * l_pde + l_mc + lambdas['ic'] * l_ic + 
              lambdas['s0'] * l_s0 + lambdas['neumann'] * l_neu)
    
    t_loss.backward()
    
    if lbfgs_iter % 100 == 0:
         print(f"L-BFGS Iteráció {lbfgs_iter}: Loss={t_loss.item():.6e} | PDE: {l_pde.item():.6e} | MC: {l_mc.item():.6e} | IC: {l_ic.item():.6e} | S0: {l_s0.item():.6e} | Neumann: {l_neu.item():.6e}")
    lbfgs_iter += 1
    
    return t_loss

model.train()
l_optimizer.step(closure)

# --- MENTÉS ÉS PLOT ---
plt.figure(figsize=(10, 5))
plt.plot(epoch_history, loss_history, 'b-', linewidth=2)
plt.yscale('log')
plt.title('Heston Asian PINN - Képzés összesített vesztesége')
plt.xlabel('Korszakok (Epochs)')
plt.ylabel('Loss (Log Scale)')
plt.grid(True, alpha=0.3)
plt.savefig('training_loss_final.png', dpi=150)
plt.show()

torch.save(model.state_dict(), 'heston_asian_pinn_model_final.pth')
print("✓ KÉSZ!")